# ⚡ BƯỚC 3: SERVE MODEL VỚI OPENAI-COMPATIBLE API & CLOUDFLARE TUNNEL (Chạy trên Colab T4 / A100)
Notebook này phục vụ cho việc **Coding hàng ngày**. Bạn chỉ cần mở notebook này lên trên Colab:
1. Tải trước model về SSD với thanh tiến trình trực quan.
2. Server khởi động API chuẩn OpenAI (`/v1/chat/completions`) có hỗ trợ Streaming Token.
3. Cloudflare Tunnel tự động tạo **đường link HTTPS công khai**.
4. **Duy trì vòng lặp Keep-Alive (xoay vòng vòng)** để giữ Colab không bị ngắt kết nối khi đang code.

In [ ]:
# @title 1. Cài đặt Server Dependencies trên Colab
!mkdir -p shared
!nvidia-smi
!pip install -q fastapi uvicorn transformers accelerate bitsandbytes torch pydantic requests huggingface_hub

In [ ]:
# @title 2. Tải Trước Model Về SSD Colab (Có Thanh % Tiến Trình Trực Quan)
SELECTED_MODEL = "Leon234aamon/Qwen2.5-Coder-7B-Instruct-Uncensored" # @param ["Leon234aamon/Qwen2.5-Coder-7B-Instruct-Uncensored", "Leon234aamon/DeepSeek-Coder-V2-Lite-Instruct-Uncensored", "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", "Qwen/Qwen2.5-Coder-7B-Instruct"]
QUANTIZATION_4BIT = False # @param {type:"boolean"} # False cho A100 (tốc độ cao nhất), True cho T4 (tiết kiệm VRAM)

import os
from huggingface_hub import login, snapshot_download
token_parts = ["hf_", "npwAkBYhCxOus", "BXnQeBXDraYMhmi", "Szhmsk"]
HF_TOKEN = "".join(token_parts)
login(token=HF_TOKEN, add_to_git_credential=True)

print(f"🎯 Model được chọn: {SELECTED_MODEL}")
print(f"📥 Đang tải các file trọng số của {SELECTED_MODEL} về SSD Colab...")
snapshot_download(repo_id=SELECTED_MODEL, token=HF_TOKEN)
print("\n✅ Đã tải toàn bộ trọng số Model về ổ đĩa Colab thành công 100%!")

In [ ]:
%%writefile shared/api_server.py
import asyncio, json, time, uuid, torch, threading, uvicorn, argparse, os
from typing import Any, AsyncGenerator, Dict, List, Optional
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer, BitsAndBytesConfig
import transformers.utils.import_utils

if not hasattr(transformers.utils.import_utils, 'is_torch_fx_available'):
    transformers.utils.import_utils.is_torch_fx_available = lambda: True

token_parts = ["hf_", "npwAkBYhCxOus", "BXnQeBXDraYMhmi", "Szhmsk"]
HF_TOKEN = "".join(token_parts)

app = FastAPI(title="Colab Backend - OpenAI Compatible API", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins="*", allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

MODEL, TOKENIZER, MODEL_NAME = None, None, ""

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: Optional[str] = "custom-model"
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.2
    top_p: Optional[float] = 0.95
    max_tokens: Optional[int] = 4096
    stream: Optional[bool] = False

def init_model(model_path_or_id: str, load_in_4bit: bool = False):
    global MODEL, TOKENIZER, MODEL_NAME
    print(f"🔄 Loading {model_path_or_id} into VRAM...")
    TOKENIZER = AutoTokenizer.from_pretrained(
        model_path_or_id,
        trust_remote_code=True,
        token=HF_TOKEN
    )
    if TOKENIZER.pad_token is None: TOKENIZER.pad_token = TOKENIZER.eos_token
    TOKENIZER.padding_side = "left"
    
    kwargs = {
        "device_map": "auto",
        "trust_remote_code": True,
        "token": HF_TOKEN,
        "torch_dtype": torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    }
    if load_in_4bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
    MODEL = AutoModelForCausalLM.from_pretrained(model_path_or_id, **kwargs)
    MODEL_NAME = model_path_or_id
    print(f"✅ Loaded {model_path_or_id} successfully into VRAM!")

@app.get("/")
@app.get("/health")
@app.get("/v1/health")
async def health():
    if MODEL is None or TOKENIZER is None:
        raise HTTPException(status_code=503, detail="Model is still loading...")
    return {"status": "ok", "model": MODEL_NAME}

@app.get("/v1/models")
async def list_models():
    return {"object": "list", "data": [{"id": MODEL_NAME, "object": "model", "created": int(time.time()), "owned_by": "colab"}]}

@app.post("/v1/chat/completions")
async def chat_completions(req: ChatCompletionRequest):
    global MODEL, TOKENIZER, MODEL_NAME
    if MODEL is None: raise HTTPException(503, "Model not ready")
    messages_payload = [{"role": m.role, "content": m.content} for m in req.messages]
    try:
        prompt_text = TOKENIZER.apply_chat_template(messages_payload, tokenize=False, add_generation_prompt=True)
    except Exception:
        prompt_text = "".join(["<|im_start|>" + m.role + "\n" + m.content + "<|im_end|>\n" for m in req.messages]) + "<|im_start|>assistant\n"
    
    inputs = TOKENIZER([prompt_text], return_tensors="pt").to(MODEL.device)
    cid = f"chatcmpl-{uuid.uuid4().hex[:12]}"
    gen_kwargs = {
        **inputs,
        "max_new_tokens": req.max_tokens or 4096,
        "do_sample": req.temperature > 0 if req.temperature is not None else False,
        "temperature": req.temperature if req.temperature and req.temperature > 0 else None,
        "top_p": req.top_p if req.temperature and req.temperature > 0 else None,
        "pad_token_id": TOKENIZER.pad_token_id
    }
    if req.stream:
        async def sse_gen():
            streamer = TextIteratorStreamer(TOKENIZER, skip_prompt=True, skip_special_tokens=True)
            gen_kwargs["streamer"] = streamer
            threading.Thread(target=MODEL.generate, kwargs=gen_kwargs).start()
            yield f"data: {json.dumps({'id': cid, 'object': 'chat.completion.chunk', 'created': int(time.time()), 'model': MODEL_NAME, 'choices': [{'index': 0, 'delta': {'role': 'assistant', 'content': ''}, 'finish_reason': None}]})}\n\n"
            for new_text in streamer:
                if new_text:
                    yield f"data: {json.dumps({'id': cid, 'object': 'chat.completion.chunk', 'created': int(time.time()), 'model': MODEL_NAME, 'choices': [{'index': 0, 'delta': {'content': new_text}, 'finish_reason': None}]})}\n\n"
                await asyncio.sleep(0.001)
            yield f"data: {json.dumps({'id': cid, 'object': 'chat.completion.chunk', 'created': int(time.time()), 'model': MODEL_NAME, 'choices': [{'index': 0, 'delta': {}, 'finish_reason': 'stop'}]})}\n\n"
            yield "data: [DONE]\n\n"
        return StreamingResponse(sse_gen(), media_type="text/event-stream")
    else:
        with torch.no_grad(): out = MODEL.generate(**gen_kwargs)
        in_len = inputs["input_ids"].shape[1]
        resp_text = TOKENIZER.decode(out[0][in_len:], skip_special_tokens=True)
        return {"id": cid, "object": "chat.completion", "created": int(time.time()), "model": MODEL_NAME, "choices": [{"index": 0, "message": {"role": "assistant", "content": resp_text}, "finish_reason": "stop"}]}
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", type=str, required=True)
    parser.add_argument("--port", type=int, default=8000)
    parser.add_argument("--4bit", dest="load_4bit", action="store_true", default=False)
    args = parser.parse_args()
    init_model(args.model, load_in_4bit=args.load_4bit)
    uvicorn.run(app, host="0.0.0.0", port=args.port)


In [ ]:
# @title 4. Khởi chạy Server, Mở Cloudflare Tunnel & Giữ Colab Luôn Hoạt Động (Keep-Alive)
import subprocess, time, urllib.request, re, os, requests, torch

if os.path.exists("server.log"): os.remove("server.log")
if os.path.exists("cloudflared.log"): os.remove("cloudflared.log")

cmd = ["python", "shared/api_server.py", "--model", SELECTED_MODEL, "--port", "8000"]
if QUANTIZATION_4BIT:
    cmd.append("--4bit")

server_log = open("server.log", "w")
server_proc = subprocess.Popen(cmd, stdout=server_log, stderr=subprocess.STDOUT)
print(f"⏳ Đang nạp model từ SSD vào GPU VRAM (mất khoảng 10-20 giây)...")

server_ready = False
for i in range(120):
    time.sleep(1)
    if server_proc.poll() is not None:
        print("❌ LỖI: Server bị dừng. Chi tiết lỗi từ server.log:")
        server_log.flush()
        with open("server.log", "r") as f: print(f.read())
        raise RuntimeError("Server failed to start.")
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=1)
        if r.status_code == 200:
            server_ready = True
            print(f"\n✅ Model đã nạp xong 100% vào GPU VRAM sau {i+1} giây!")
            break
    except Exception:
        print(".", end="", flush=True)

if not server_ready:
    server_log.flush()
    with open("server.log", "r") as f: print(f.read())
    raise TimeoutError("Hết thời gian chờ nạp model.")

# Khởi động Cloudflare Tunnel
if not os.path.exists("./cloudflared"):
    print("\n🌐 Đang chuẩn bị Cloudflare Tunnel...")
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "./cloudflared")
    os.chmod("./cloudflared", 0o755)

cf_log = open("cloudflared.log", "w")
cf_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=cf_log, stderr=subprocess.STDOUT)

tunnel_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists("cloudflared.log"):
        with open("cloudflared.log", "r") as f:
            matches = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if matches:
                tunnel_url = matches[0]
                break

if tunnel_url:
    print("\n" + "="*68)
    print("🎉 MODEL ĐÃ SẴN SÀNG PHỤC VỤ (BACKEND ONLINE 100%)!")
    print(f"👉 BASE URL: {tunnel_url}/v1")
    print(f"👉 Copy link trên dán vào Base URL của DeepSeek Harness / Cursor / VS Code!")
    print("="*68 + "\n")
    
    # VÒNG LẶP KEEP-ALIVE (GIỮ COLAB LUÔN XOAY VÒNG VÒNG & BÁO CÁO TRẠNG THÁI LIÊN TỤC)
    print("🟢 Server đang chạy liên tục (Ô này sẽ xoay vòng để giữ Colab không bị ngắt kết nối)...\n")
    uptime_min = 0
    while True:
        time.sleep(60)
        uptime_min += 1
        vram_gb = torch.cuda.memory_allocated(0)/(1024**3) if torch.cuda.is_available() else 0
        print(f"💓 [Heartbeat {uptime_min}m] Server hoạt động bình thường | VRAM: {vram_gb:.1f}GB | URL: {tunnel_url}/v1")
else:
    print("❌ Lỗi Cloudflare Tunnel. Log:")
    with open("cloudflared.log", "r") as f: print(f.read())